# 07 · Avaliação de Impacto — Projeto Vértice (Vértice Retail)

**Papel deste notebook:** substitui o antigo `metodologia_impacto.md`. Onde o `.md` era
texto com números digitados à mão, aqui **cada impacto é calculado em código**, puxado da
fonte única (`06_numeros_canonicos.ipynb` → `outputs/numeros_canonicos.json`), com a
coluna-fonte e a fórmula explícitas. É a **base financeira da dashboard HTML**.

**O que este notebook adiciona ao número:** a régua não é só R$. Cada frente carrega, de
forma estruturada, **confiança** (Alta/Média/Baixa), **natureza** (perda medida · receita
em risco · custo recorrente · economia · exposição · teto · multiplicador), **tratamento**
(líquido vs teto vs premissa) e **sobreposição** (com quais outras frentes se cruza). Isso
existe porque **os números da régua não estão sob as mesmas condições** — uma barra de
"risco" (ruptura) não é comparável 1:1 com uma de "perda medida" (margem não realizada).

**Relação com os outros notebooks:** os vereditos vêm dos testes do `03`; a janela, do `01`
(§6); o enquadramento de slide, do `04`; e todo R$ é o valor canônico do `06`. Este
notebook consolida — não recalcula uma análise nova.

**Régua comum:** R$/ano (anualizado, sem projeção ×2,88) **e** % da margem de contribuição
**realizada** anual (~R$ 7,2 mi) — "quanto do lucro que de fato entra cada frente
representa".


In [1]:
import json
from pathlib import Path

CANON = json.loads(Path('outputs/numeros_canonicos.json').read_text(encoding='utf-8'))['metricas']
def val(chave):  # puxa o valor canônico (fonte única de verdade)
    return CANON[chave]['valor']

DEN = val('margem_realizada_ano')   # denominador da régua
print(f"Denominador da régua (margem realizada anual): R$ {DEN:,.0f}/ano")
print(f"Métricas canônicas disponíveis: {len(CANON)}")

def pct_margem(rs_ano):
    return round(100*rs_ano/DEN, 1)

Denominador da régua (margem realizada anual): R$ 7,212,700/ano
Métricas canônicas disponíveis: 98


## 1. As 14 hipóteses da árvore MECE

A árvore de hipóteses decompõe a pergunta-topo do case ("como usar dados e IA para
recuperar rentabilidade, eficiência e qualidade de decisão em 90 dias?") em **6 segmentos**
mutuamente exclusivos, e cada segmento em hipóteses testáveis. Aqui registramos o veredito
de cada uma (dos testes do notebook 03) e a qual frente de impacto ela se liga.

In [2]:
HIPOTESES = [
 # id, segmento, hipótese, pergunta de negócio, veredito, impacto_id
 ('H1','Margem','O mix de produtos migrou para categorias menos rentáveis','A margem cai por mudança de mix de categoria?','Refutada',None),
 ('H2','Margem','A política de desconto está corroendo a margem','O que corrói a margem por pedido?','Confirmada','desconto'),
 ('H3','Margem','O custo do fornecedor subiu e não foi repassado ao preço','Há inflação de custo não repassada?','Refutada',None),
 ('H4','Marketing','A verba está mal distribuída entre os canais','Onde realocar investimento de marketing?','Não testável','marketplace_frete'),
 ('H5','Marketing','Existe um gargalo de conversão no funil','Há um gargalo de conversão a destravar?','Não testável',None),
 ('H6','Marketing','Alguns canais atraem clientes de baixo valor','Onde concentrar retenção por qualidade de aquisição?','Não testável',None),
 ('H7','Operações','A falta de estoque nos produtos de maior giro destrói receita','Ruptura em alto giro está custando receita?','Parcial','ruptura'),
 ('H8','Operações','Prazo de fornecedor e entrega lenta geram indisponibilidade','Vale investir em logística/fornecedor?','Refutada',None),
 ('H9','Operações','A devolução destrói mais margem do que a venda que desfaz','Quanto da margem some depois da venda?','Confirmada','margem_nao_realizada'),
 ('H10','Atendimento','O volume de chamados é sintoma da operação, não demanda de suporte','Contratar suporte ou consertar a origem?','Confirmada','falha_operacional'),
 ('H11','Atendimento','Parte relevante dos chamados dispensa um atendente','Dá para automatizar sem perder qualidade?','Confirmada','chatbot'),
 ('H12','Atendimento','O atendimento ruim antecede a perda do cliente','Atendimento ruim causa churn?','Refutada',None),
 ('H13','Gestão','A informação disponível não está virando decisão','A empresa reage aos sinais que já tem?','Confirmada','multiplicador'),
 ('H14','Produtividade','A triagem manual de chamados é automatizável','Um classificador resolve a triagem?','Com ressalva',None),
]
from collections import Counter
placar = Counter(h[4] for h in HIPOTESES)
print('Placar dos vereditos:', dict(placar))

Placar dos vereditos: {'Refutada': 4, 'Confirmada': 5, 'Não testável': 3, 'Parcial': 1, 'Com ressalva': 1}


## 2. Frentes com valor em R$ — cada uma calculada e condicionada

Cada frente puxa o valor canônico do `06`, converte para % da régua e recebe as etiquetas
de condição. A ordem de leitura segue Fato → Causa → Recomendação.

### 2.1 Margem não realizada (H9) — o maior número, e uma perda medida

**Premissa:** `margem_contribuicao` conta pedidos que nunca viraram caixa (devolvidos,
cancelados, aguardando pagamento). A margem realizada recalcula só os aprovados e não
devolvidos, sobre a mesma receita. **Fonte:** Teste 15 (nb03), canônico `margem_gap_rs_ano`.
**Ressalva:** a base não tem frete reverso; o único custo afundado adicional é o frete de
saída de devolvidos (R$ 51 mil na janela), não incluído no headline.

In [3]:
impactos = []
impactos.append(dict(
    id='margem_nao_realizada', hipotese_id='H9', segmento='Operações',
    titulo='Margem que nunca virou caixa (devolução, cancelamento, pagamento pendente)',
    rs_ano=val('margem_gap_rs_ano'), pct_margem=pct_margem(val('margem_gap_rs_ano')),
    confianca='Alta', natureza='perda medida', tratamento='valor líquido (recálculo contábil direto)',
    sobreposicao=['desconto'],
    premissa='pedidos aprovados e não devolvidos, sobre a receita líquida total',
    formula='margem_contabil − margem_realizada (13m) × 365/dias',
    fonte_canonica=['margem_gap_rs_13m','margem_gap_rs_ano','margem_realizada_pct'],
    ressalva=f"Base sem frete reverso; frete de saída afundado em devolvidos = R$ {val('frete_afundado_devolvidos_rs_13m'):,.0f} (não somado ao headline).",
))
print(impactos[-1]['titulo'], '→ R$ %.0f/ano  (%.1f%% da margem realizada)' % (impactos[-1]['rs_ano'], impactos[-1]['pct_margem']))

Margem que nunca virou caixa (devolução, cancelamento, pagamento pendente) → R$ 2399375/ano  (33.3% da margem realizada)


### 2.2 Desconto sem contrapartida (H2) — um teto, não um líquido

**Premissa:** o notebook 04 mostra que unidades por pedido **não sobem** com o desconto
(amplitude de 0,14 unidade entre faixas). Logo cada real de desconto é margem cedida 1:1 —
mas isso é um **teto** de recuperação (a recomendação é teto por faixa, não zerar desconto).
**Sobreposição:** parte do desconto está em pedidos que também foram devolvidos/cancelados
(conta com a margem não realizada) — **não somar as duas**.

In [4]:
impactos.append(dict(
    id='desconto', hipotese_id='H2', segmento='Margem',
    titulo='Margem cedida em desconto sem ganho de volume',
    rs_ano=val('desconto_total_rs_ano'), pct_margem=pct_margem(val('desconto_total_rs_ano')),
    confianca='Alta', natureza='teto', tratamento='teto 1:1 (desconto = margem cedida)',
    sobreposicao=['margem_nao_realizada'],
    premissa='soma do desconto concedido; unidades por pedido não sobem com desconto (nb04)',
    formula='soma(desconto_reais) 13m × 365/dias',
    fonte_canonica=['desconto_total_rs_ano','unidades_amplitude_faixas','desconto_spread_pp'],
    ressalva=f"É teto, não líquido — a recomendação é teto por faixa (a de 25%+ tem {val('pedidos_faixa_top')} pedidos a {val('margem_faixa_top_pct')}% de margem), não eliminar o desconto.",
))
print(impactos[-1]['titulo'], '→ R$ %.0f/ano  (%.1f%%)' % (impactos[-1]['rs_ano'], impactos[-1]['pct_margem']))

Margem cedida em desconto sem ganho de volume → R$ 1531877/ano  (21.2%)


### 2.3 Vão de frete do Marketplace (H4/canal) — líquido, alta confiança

**Premissa:** o Marketplace tem margem menor não por desconto (igual à média) mas por
**frete**: 100% dos pedidos pagam frete, contra 12–22% nos demais. **Fonte:** Teste 2.1
(nb03), canônico `mkt_gap_rs_ano` (versão **agregada**). **Nota:** a margem mediana por
pedido (49,6%) e o frete médio por pedido (9,7%) são estatísticas de teste — o R$ usa a
versão agregada (margem 51,4%, frete 4,9%).

In [5]:
impactos.append(dict(
    id='marketplace_frete', hipotese_id='H4', segmento='Marketing',
    titulo='Vão de margem do canal Marketplace, causado por frete',
    rs_ano=val('mkt_gap_rs_ano'), pct_margem=pct_margem(val('mkt_gap_rs_ano')),
    confianca='Alta', natureza='valor recuperável', tratamento='valor líquido (gap agregado × receita do canal)',
    sobreposicao=[],
    premissa='margem agregada do Marketplace vs demais canais, sobre a receita do canal',
    formula='(margem_demais − margem_mkt) agregada × receita_liq_mkt (13m) × 365/dias',
    fonte_canonica=['mkt_gap_pp','mkt_gap_rs_ano','mkt_frete_pct_agregado'],
    ressalva='Alavanca é renegociação logística (comercial), não corte de desconto. R$ usa métrica agregada; 49,6%/9,7% são estatísticas de teste por pedido.',
))
print(impactos[-1]['titulo'], '→ R$ %.0f/ano  (%.1f%%)' % (impactos[-1]['rs_ano'], impactos[-1]['pct_margem']))

Vão de margem do canal Marketplace, causado por frete → R$ 139418/ano  (1.9%)


### 2.4 Ruptura em curva A (H7) — receita em RISCO, não perda incorrida

**Premissa:** 417 SKUs de alto giro em ruptura/crítico. **Confiança Média** e natureza
distinta das demais: a base de estoque é uma foto, sem histórico de dias sem estoque —
então isto é **receita exposta se a ruptura persistir**, não perda já ocorrida. Não é
comparável 1:1 com a margem não realizada.

In [6]:
impactos.append(dict(
    id='ruptura', hipotese_id='H7', segmento='Operações',
    titulo='Receita em risco por ruptura nos produtos de maior giro',
    rs_ano=val('ruptura_curvaA_receita_ano'), pct_margem=pct_margem(val('ruptura_curvaA_receita_ano')),
    confianca='Média', natureza='receita em risco', tratamento='extrapolação de risco (não perda medida)',
    sobreposicao=[],
    premissa='receita histórica dos SKUs curva A em ruptura/crítico, projetada ao ano',
    formula='receita_hist(SKUs críticos A) 13m × 365/dias',
    fonte_canonica=['ruptura_curvaA_qtd','ruptura_curvaA_receita_dia','ruptura_curvaA_receita_ano'],
    ressalva='RISCO, não perda: sem histórico de dias sem estoque, é receita exposta se a ruptura persistir o ano — não somar com perdas medidas.',
))
print(impactos[-1]['titulo'], '→ R$ %.0f/ano  (%.1f%%, RISCO)' % (impactos[-1]['rs_ano'], impactos[-1]['pct_margem']))

Receita em risco por ruptura nos produtos de maior giro → R$ 2098193/ano  (29.1%, RISCO)


### 2.5 Custo de falha operacional (H10) e economia do ChatBot (H11) — atendimento

**H10:** 60% dos chamados nascem de falha da operação — R$ 107 mil/ano de custo recorrente
endereçável na origem (logística). **H11:** o ChatBot já entrega CSAT igual ao humano
(3,27 vs 3,23) a 11% do custo — migrar "onde está meu pedido" economiza R$ 46 mil/ano, sem
tecnologia nova. Ambos vêm de 36 meses de atendimento (÷3), **sem projeção**.

In [7]:
impactos.append(dict(
    id='falha_operacional', hipotese_id='H10', segmento='Atendimento',
    titulo='Custo de atendimento gerado por falha operacional anterior',
    rs_ano=val('atend_custo_falha_ano'), pct_margem=pct_margem(val('atend_custo_falha_ano')),
    confianca='Alta', natureza='custo recorrente', tratamento='custo endereçável na origem',
    sobreposicao=[],
    premissa='tickets de atraso/defeito/pagamento (falha operacional), custo operacional',
    formula='soma(custo dos tickets de falha) 36m ÷ 3',
    fonte_canonica=['atend_custo_falha_ano','atend_pct_falha_volume','atend_custo_total_ano'],
    ressalva=f"Reduzível atacando a origem (logística), não o atendimento. Parte de um custo total de R$ {val('atend_custo_total_ano'):,.0f}/ano.",
))
impactos.append(dict(
    id='chatbot', hipotese_id='H11', segmento='Atendimento',
    titulo='Economia migrando volume simples para o ChatBot já existente',
    rs_ano=val('chatbot_economia_ano'), pct_margem=pct_margem(val('chatbot_economia_ano')),
    confianca='Alta', natureza='economia', tratamento='economia observada (sem tecnologia nova)',
    sobreposicao=['falha_operacional'],
    premissa="tickets 'onde está meu pedido' fora do ChatBot, ao custo do ChatBot (R$2/ticket)",
    formula='(custo_atual − nº×R$2,00) 36m ÷ 3',
    fonte_canonica=['chatbot_tickets_migraveis','chatbot_economia_ano','chatbot_csat','humano_csat'],
    ressalva=f"CSAT do ChatBot ({val('chatbot_csat')}) ≥ humano ({val('humano_csat')}); sobrepõe parcialmente ao custo de falha operacional.",
))
for i in impactos[-2:]:
    print(i['titulo'], '→ R$ %.0f/ano  (%.1f%%)' % (i['rs_ano'], i['pct_margem']))

Custo de atendimento gerado por falha operacional anterior → R$ 106703/ano  (1.5%)
Economia migrando volume simples para o ChatBot já existente → R$ 46043/ano  (0.6%)


## 3. Frentes sem R$ — o porquê da ausência (também é achado)

Uma consultoria séria declara por que **não** pôs um número, em vez de inventar um. Cada
uma vira recomendação de processo, instrumentação ou é uma exposição (não perda).

In [8]:
sem_rs = [
 dict(hipotese_id='H1', titulo='Mix de categoria', motivo='0,34 p.p. entre a melhor e a pior categoria — não é driver.', natureza='refutada'),
 dict(hipotese_id='H3', titulo='Repasse de custo de fornecedor', motivo='CMV estável em 43,8% da receita; custo cadastrado não bate com o praticado (corr 0,005).', natureza='refutada'),
 dict(hipotese_id='H5', titulo='Gargalo de funil', motivo='marketing declara 4.153× os pedidos reais; funil não auditável.', natureza='não testável'),
 dict(hipotese_id='H6', titulo='Canais de baixo valor', motivo='346 IDs para 27.758 pedidos; métricas por cliente bloqueadas em vendas.', natureza='não testável / exposição',
      exposicao_rs=val('rfm_ltv_exposto'), exposicao_label=f"R$ {val('rfm_ltv_exposto')/1e6:.1f} mi de LTV em {val('rfm_clientes_exposto')} clientes de segmentos em risco (exposição na base de clientes, não perda)"),
 dict(hipotese_id='H8', titulo='Investimento logístico geral', motivo='entrega lenta não gera mais devolução (1,3 p.p.); efeito de fornecedor lento é pequeno (V=0,049).', natureza='refutada'),
 dict(hipotese_id='H12', titulo='Atendimento ruim → churn', motivo='sem associação estatística (qui-quadrado p=0,11); churn sub-representado no grupo de má experiência.', natureza='refutada'),
 dict(hipotese_id='H13', titulo='Painel de decisão comercial', motivo='4 de 4 sinais disponíveis são ignorados; valor é destravar as outras frentes de forma recorrente.', natureza='multiplicador'),
 dict(hipotese_id='H14', titulo='Classificador de triagem', motivo='acurácia honesta 43% vs baseline 30% — perto do acaso com 30 frases fixas; sem base para R$.', natureza='com ressalva'),
]
for s in sem_rs: print(s['hipotese_id'], '·', s['titulo'], '—', s['natureza'])

H1 · Mix de categoria — refutada
H3 · Repasse de custo de fornecedor — refutada
H5 · Gargalo de funil — não testável
H6 · Canais de baixo valor — não testável / exposição
H8 · Investimento logístico geral — refutada
H12 · Atendimento ruim → churn — refutada
H13 · Painel de decisão comercial — multiplicador
H14 · Classificador de triagem — com ressalva


## 4. Régua comparável e a regra de "não somar"

Todas as frentes com R$ na mesma régua: R$/ano **e** % da margem realizada anual. **Não
somar**: margem não realizada e desconto se sobrepõem (parte do desconto está em pedidos
devolvidos/cancelados); ChatBot e custo de falha também. O número defensável numa arguição
é **cada frente isolada**, com sua etiqueta de natureza/confiança.

In [9]:
impactos_sorted = sorted(impactos, key=lambda x: -x['rs_ano'])
print(f"{'Frente':52} {'R$/ano':>13} {'% margem':>9}  {'Confiança':9} {'Natureza'}")
for i in impactos_sorted:
    print(f"{i['titulo'][:50]:52} {i['rs_ano']:>13,.0f} {i['pct_margem']:>8.1f}%  {i['confianca']:9} {i['natureza']}")

# teto conservador: só perdas/custos/economias MEDIDOS de alta confiança, sem sobreposição dupla.
# margem não realizada já é o guarda-chuva da margem; somamos a ela as frentes operacionais
# distintas de alta confiança (marketplace, falha operacional, chatbot). Desconto (teto,
# sobrepõe) e ruptura (risco, média) ficam FORA do teto somável.
alta_distintas = ['margem_nao_realizada','marketplace_frete','falha_operacional','chatbot']
teto = sum(i['rs_ano'] for i in impactos if i['id'] in alta_distintas)
print(f"\nTeto conservador (frentes de alta confiança, sem sobreposição): R$ {teto:,.0f}/ano ({100*teto/DEN:.1f}% da margem realizada)")
print('Desconto (teto, sobrepõe) e ruptura (risco, média confiança) ficam de fora do somável.')

Frente                                                      R$/ano  % margem  Confiança Natureza
Margem que nunca virou caixa (devolução, cancelame       2,399,375     33.3%  Alta      perda medida
Receita em risco por ruptura nos produtos de maior       2,098,193     29.1%  Média     receita em risco
Margem cedida em desconto sem ganho de volume            1,531,877     21.2%  Alta      teto
Vão de margem do canal Marketplace, causado por fr         139,418      1.9%  Alta      valor recuperável
Custo de atendimento gerado por falha operacional          106,703      1.5%  Alta      custo recorrente
Economia migrando volume simples para o ChatBot já          46,043      0.6%  Alta      economia

Teto conservador (frentes de alta confiança, sem sobreposição): R$ 2,691,539/ano (37.3% da margem realizada)
Desconto (teto, sobrepõe) e ruptura (risco, média confiança) ficam de fora do somável.


## 5. Exportação — `outputs/impacto.json` (base financeira da HTML)

In [10]:
from collections import Counter
placar = dict(Counter(h[4] for h in HIPOTESES))
payload = {
  '_meta': {'gerado_por':'07_avaliacao_impacto.ipynb', 'fonte_numeros':'06_numeros_canonicos.ipynb',
            'projecao_base_completa': False},
  'regua': {'denominador_rs_ano': round(DEN,2), 'label':'margem de contribuição realizada anual'},
  'placar': placar,
  'hipoteses': [dict(id=h[0], segmento=h[1], hipotese=h[2], pergunta=h[3], veredito=h[4], impacto_id=h[5]) for h in HIPOTESES],
  'impactos': impactos_sorted,
  'sem_rs': sem_rs,
  'teto_conservador_rs_ano': round(teto,2),
  'nao_somar': 'Margem não realizada e desconto se sobrepõem; ChatBot e custo de falha também. O número defensável é cada frente isolada. Ruptura é risco (não perda) e desconto é teto (não líquido) — fora do somável.',
}
Path('outputs/impacto.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('impacto.json exportado:', len(impactos), 'frentes com R$;', len(sem_rs), 'sem R$;', 'placar', placar)

impacto.json exportado: 6 frentes com R$; 8 sem R$; placar {'Refutada': 4, 'Confirmada': 5, 'Não testável': 3, 'Parcial': 1, 'Com ressalva': 1}
